In [ ]:
import json
import os
import time
from dashscope import MultiModalConversation

def GetPositions(dir):
    return [(item, os.path.join(dir, item).replace("\\", "/")) for item in os.listdir(dir)]

def GetImages(position_path):
    return [os.path.join(position_path, item).replace("\\", "/") for item in os.listdir(position_path)]

def GetInputContent(position_path):
    images = GetImages(position_path)
    prompt = "<div>图示为某个地方四个方位的街景图组成的全景图。请给出安全或不安全的判断，并为该处街景的安全感打分。从0至100评分，分数越高，说明人在其中越感到安全。注意，60分是安全/不安全的阈值。</div>" + \
         "<div>" + \
         "安全感打分时，需要综合考虑以下三个方面：" + \
         "<span>1. 人在此处人身安全受到威胁的可能性（如抢劫、盗窃等犯罪行为、交通事故等安全事件、重物坠落砸伤等意外事故）；</span>" + \
         "<span>2. 人在此处的紧张、焦虑和恐惧等负面情绪的程度；</span>" + \
         "<span>3. 人在此处行走时有多大可能性因为这些可能性和负面情绪而更改路线。</span>" + \
         "</div>" + \
         "<div>安全感的定义为：人对所处环境中不会存在危险因素的确定感和可控制感，即在多大程度上觉得所处环境中不会存在危险因素以及不会产生以上负面情绪。</div>" + \
         "<div>请以json的格式返回结果；json中分数请以数字而非字符呈现，命名为score；具体理由命名为reason。</div>"
    content = [{'image': f"file://{image}"} for image in images]
    content.append({'text': prompt})
    return content

def GetSystemRole(role):
    if role == "female":
        return "You are a female resident."
    if role == "male":
        return "You are a male resident."
    if role == "local":
        return "You are a resident who has a hukou of the city the streetviews located (i.e., Xi'an)."
    if role == "non-local":
        return "You are a resident who does not have a hukou of the city the streetviews located (i.e., Xi'an)."
    if role == "experience":
        return "You are a resident who has previously experienced a crime or accident, such as a robbery theft, or traffic accident, that threatened your personal safety and left you feeling tense, fearful, and anxious. "
    if role == "non-experience":
        return "You are a resident who has never experienced a crime or accident, such as a robbery theft, or traffic accident, that threatened your personal safety and left you feeling tense, fearful, and anxious. "

def GetResponse(position_path, role, api_key):
    content = GetInputContent(position_path)
    role_content = GetSystemRole(role)
    messages = [{"role": "system",
                 "content": [{"text": role_content}]},
                {"role": "user",
                 "content": content}]
    return MultiModalConversation.call(
        api_key=api_key,
        model='qwen-vl-max-latest',
        messages = messages,
        temperature = 0)

def SavePositionResponse(response, item, role, round):
    with open(f"Results/response_{item}_{role}_{round}.json", 'w') as f:
        json.dump(response, f, ensure_ascii = False)

def CheckPositionResponse(item, role, round):
    if os.path.exists(f"Results/response_{item}_{role}_{round}.json"):
        return True

def WorkOnPosition(position, role, api_key):
    position_item, position_path = position[0], position[1]
    for i in range(1, 3):
        if CheckPositionResponse(position_item, role, i):
            print(f"Position {position_item} role {role} round {i} has been skipped.")
            continue
        response = GetResponse(position_path, role, api_key)
        SavePositionResponse(response, position_item, role, i)
        print(f"Position {position_item} role {role} round {i} has been processed.")

data_root = "Data/StreetViews"
api_key = "[API KEY HERE]"
positions = GetPositions(data_root)
for position in positions:
    for role in ["female", "male", "local", "non-local", "experience", "non-experience"]:
        try:
            WorkOnPosition(position, role, api_key)
        except:
            time.sleep(30)

Position 1 role female round 1 has been skipped.
Position 1 role female round 2 has been skipped.
Position 1 role male round 1 has been skipped.
Position 1 role male round 2 has been skipped.
Position 1 role local round 1 has been skipped.
Position 1 role local round 2 has been skipped.
Position 1 role non-local round 1 has been skipped.
Position 1 role non-local round 2 has been skipped.
Position 1 role experience round 1 has been skipped.
Position 1 role experience round 2 has been skipped.
Position 1 role non-experience round 1 has been skipped.
Position 1 role non-experience round 2 has been skipped.
Position 10 role female round 1 has been skipped.
Position 10 role female round 2 has been skipped.
Position 10 role male round 1 has been skipped.
Position 10 role male round 2 has been skipped.
Position 10 role local round 1 has been skipped.
Position 10 role local round 2 has been skipped.
Position 10 role non-local round 1 has been skipped.
Position 10 role non-local round 2 has be